----
# ML-DSA
----

In [4]:
import time
import statistics
import csv
import oqs
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization

# Configuration
runs = 20
message = b"Benchmarking Post-Quantum Cryptographic Algorithms"

# ==========================================
# PART 1: ML-DSA-44 (AES-128 Equivalent)
# ==========================================
print("Benchmarking ML-DSA-44...")
ml_dsa_algo = "ML-DSA-44"
ml_keypair_times = []
ml_signing_times = []

with oqs.Signature(ml_dsa_algo) as signer:
    ml_pk_size = signer.details['length_public_key']
    ml_sk_size = signer.details['length_secret_key']
    ml_sig_size = signer.details['length_signature']
    
    # ML-DSA Keypair
    for _ in range(runs):
        start_time = time.perf_counter()
        _ = signer.generate_keypair()
        ml_keypair_times.append((time.perf_counter() - start_time) * 1000)
        
    # ML-DSA Signing
    for _ in range(runs):
        start_time = time.perf_counter()
        _ = signer.sign(message)
        ml_signing_times.append((time.perf_counter() - start_time) * 1000)

# ==========================================
# PART 2: RSA-3072 (AES-128 Equivalent)
# ==========================================
print("Benchmarking RSA-3072...")
rsa_key_size = 3072
rsa_keypair_times = []
rsa_signing_times = []

# RSA Keypair
rsa_private_keys = []
for _ in range(runs):
    start_time = time.perf_counter()
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=rsa_key_size,
    )
    rsa_keypair_times.append((time.perf_counter() - start_time) * 1000)
    rsa_private_keys.append(private_key)

# We use the last generated key to measure sizes
rsa_pk = rsa_private_keys[-1].public_key()
rsa_pk_size = len(rsa_pk.public_bytes(serialization.Encoding.DER, serialization.PublicFormat.SubjectPublicKeyInfo))
rsa_sk_size = len(rsa_private_keys[-1].private_bytes(serialization.Encoding.DER, serialization.PrivateFormat.PKCS8, serialization.NoEncryption()))

# RSA Signing
for i in range(runs):
    current_key = rsa_private_keys[i]
    start_time = time.perf_counter()
    signature = current_key.sign(
        message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256()
    )
    rsa_signing_times.append((time.perf_counter() - start_time) * 1000)
    rsa_sig_size = len(signature) # Will be exactly 384 bytes for 3072-bit

# ==========================================
# PART 3: Export Results
# ==========================================

def write_results(algo_name, pk_size, sk_size, sig_size, kp_times, sig_times, txt_filename, csv_filename):
    # Calculate stats
    mean_kp, min_kp, max_kp = statistics.mean(kp_times), min(kp_times), max(kp_times)
    mean_sig, min_sig, max_sig = statistics.mean(sig_times), min(sig_times), max(sig_times)

    # Write TXT
    txt_content = f"""{algo_name} Benchmark Results
=========================
Benchmarking {algo_name} keypair generation...
Algorithm: {algo_name}
Runs: {runs}
Mean keypair time: {mean_kp:.3f} ms
Min keypair time: {min_kp:.3f} ms
Max keypair time: {max_kp:.3f} ms
Times (ms): {kp_times}

Benchmarking {algo_name} signing...
Algorithm: {algo_name}
Runs: {runs}
Mean signing time: {mean_sig:.3f} ms
Min signing time: {min_sig:.3f} ms
Max signing time: {max_sig:.3f} ms
Signing times (ms): {sig_times}

Measuring {algo_name} size metrics...
Algorithm: {algo_name}
Public key size: {pk_size} bytes
Secret key size: {sk_size} bytes
Signature size: {sig_size} bytes
"""
    with open(txt_filename, "w") as f:
        f.write(txt_content)

    # Write CSV
    with open(csv_filename, "w", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["metric", "run", "time_ms"])
        for i, t in enumerate(kp_times, 1):
            writer.writerow(["keypair_generation", i, t])
        for i, t in enumerate(sig_times, 1):
            writer.writerow(["signing", i, t])

# Export ML-DSA
write_results(ml_dsa_algo, ml_pk_size, ml_sk_size, ml_sig_size, ml_keypair_times, ml_signing_times, "ml_dsa_benchmark_results.txt", "ml_dsa_timing.csv")

# Export RSA-3072
write_results("RSA-3072", rsa_pk_size, rsa_sk_size, rsa_sig_size, rsa_keypair_times, rsa_signing_times, "rsa_3072_benchmark_results.txt", "rsa_3072_timing.csv")

print("Benchmarking complete. All 4 files have been generated.")

Benchmarking ML-DSA-44...
Benchmarking RSA-3072...
Benchmarking complete. All 4 files have been generated.
